# 02 — Limpieza y Transformación de Datos
**Proyecto:** OpenFarma — Sistema de Alerta Temprana de Desabastecimiento Farmacéutico  

Este notebook documenta el pipeline de limpieza aplicado a los datos del CUM y del INVIMA:
- Normalización de DCIs (Denominación Común Internacional)
- Estandarización de formas farmacéuticas y vías de administración
- Construcción de grupos de equivalencia terapéutica
- Limpieza de alertas INVIMA desde PDF

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')

DB_PATH = Path('../backend/openfarma.db')
conn = sqlite3.connect(DB_PATH)
print(f'Conectado. Verificando tablas...')

tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tablas['name'].tolist())

## 1. Reto: datos del CUM sin normalizar

El CUM (API Socrata) devuelve los datos tal como los cargó el INVIMA, con inconsistencias comunes:
- **Nombres de principios activos:** mezcla de nombres comerciales, genéricos, en inglés y en español
- **Codificación ATC:** errores en el nivel 7 para medicamentos combinados
- **Formas farmacéuticas:** múltiples formas de escribir lo mismo (TABLETA RECUBIERTA / TAB RECUBIERTA / T.REC.)
- **Concentraciones:** unidades mezcladas (mg, mcg, UI, %, mg/mL)

In [ ]:
# Cargar muestra cruda de la API (tal como llega de Socrata)
cum_raw = pd.read_sql("""
    SELECT expediente_cum, consecutivo_cum, nombre_comercial_norm,
           principios_dci, tipo_formula, forma_normalizada,
           via_normalizada, atc_normalizado, dosis_total_mg, concentracion_mg_ml,
           estado_cum, titular_registro
    FROM cum_normalizado
    LIMIT 5000
""", conn)

print(f'Muestra: {len(cum_raw):,} registros')
print(f'\nValores nulos por columna:')
nulos = cum_raw.isnull().sum()
print(nulos[nulos > 0].to_string())

## 2. Normalización de principios activos (DCI)

Se implementó una tabla de sinónimos (`_SINONIMOS` en `etl/transformacion.py`) con ~420 entradas que mapea nombres comerciales y variantes lingüísticas al INN (International Nonproprietary Name) de la OMS en español.

In [ ]:
# Ejemplos de normalización aplicada
ejemplos_normalizacion = {
    'Entrada original (CUM)': [
        'ACETAMINOPHEN', 'DIPYRONE', 'ALBUTEROL', 'EPINEPHRINE',
        'CIPROFLOXACIN', 'IBUPROFEN', 'METRONIDAZOLE', 'OMEPRAZOLE'
    ],
    'DCI normalizado (INN-ES)': [
        'PARACETAMOL', 'METAMIZOL', 'SALBUTAMOL', 'ADRENALINA',
        'CIPROFLOXACINO', 'IBUPROFENO', 'METRONIDAZOL', 'OMEPRAZOL'
    ],
    'Regla aplicada': [
        'Nombre colombiano → INN', 'Nombre colombiano → INN',
        'Nombre EN → ES', 'Nombre EN → ES',
        'Sufijo -in → -ino (fluoroquinolona)', 'Sufijo -en → -eno',
        'Sufijo -ole → -ol', 'Sufijo -ole → -ol'
    ]
}

df_norm = pd.DataFrame(ejemplos_normalizacion)
print('Ejemplos de normalización DCI:')
print(df_norm.to_string(index=False))

In [ ]:
# Verificar calidad de la normalización actual
cum_all = pd.read_sql("SELECT principios_dci, tipo_formula FROM cum_normalizado WHERE estado_cum='Activo'", conn)

# Contar productos por número de principios activos
def contar_dcis(json_str):
    try:
        return len(json.loads(json_str or '[]'))
    except:
        return 0

cum_all['n_dcis'] = cum_all['principios_dci'].apply(contar_dcis)

dist = cum_all['n_dcis'].value_counts().sort_index()
print('Distribución por número de principios activos:')
for n, count in dist.items():
    barra = '█' * (count // 500)
    print(f'  {n} PA: {count:6,} productos  {barra}')

## 3. Normalización de grupos de equivalencia

Los productos se agrupan en grupos terapéuticos según: **DCI + vía de administración + forma farmacéutica + concentración**.

El proceso involucró 104 rondas de auditoría para corregir errores específicos (radiofármacos, vacunas, factores de coagulación, etc.).

In [ ]:
grupos = pd.read_sql("""
    SELECT id, dci_key, grupo_via, concentracion_norm,
           json_array_length(cum_ids) as n_productos
    FROM grupos_equivalencia
""", conn)

print(f'Total grupos de equivalencia: {len(grupos):,}')
print(f'Grupos con concentración definida: {(grupos.concentracion_norm != "SIN_CONCENTRACION").sum():,}')
print(f'Grupos SIN_CONCENTRACION (vacunas/biológicos): {(grupos.concentracion_norm == "SIN_CONCENTRACION").sum():,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grupos por vía
via_counts = grupos['grupo_via'].value_counts()
via_counts.plot(kind='bar', ax=axes[0], color='#2563eb', edgecolor='white')
axes[0].set_title('Grupos de equivalencia por vía/forma', fontsize=11)
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

# Distribución de productos por grupo
grupos['n_productos'].clip(upper=30).hist(bins=30, ax=axes[1], color='#2563eb', edgecolor='white')
axes[1].set_title('Productos por grupo (truncado en 30)', fontsize=11)
axes[1].set_xlabel('Número de productos en el grupo')
axes[1].set_ylabel('Número de grupos')

plt.tight_layout()
plt.show()

print(f'\nGrupos con 1 producto (singletons): {(grupos.n_productos == 1).sum():,}')
print(f'Grupo más grande: {grupos.n_productos.max()} productos')
print(f'Mediana de productos por grupo: {grupos.n_productos.median():.0f}')

## 4. Limpieza de alertas INVIMA

Los PDFs del INVIMA presentan varios problemas de parseo:
- ATC pegado al nombre del medicamento (e.g., `CICLOFOSFAMIDAL01AA01`)
- Campos `forma` y `concentración` fragmentados en entradas largas
- Texto de pie de tabla mezclado con datos

In [ ]:
invima = pd.read_sql("""
    SELECT mes, anio, principio_activo, forma, concentracion,
           estado, atc
    FROM invima_seguimiento
""", conn)

print(f'Total registros INVIMA cargados: {len(invima):,}')
print(f'\nValores nulos en campos clave:')
print(f'  principio_activo nulo: {invima.principio_activo.isnull().sum()}')
print(f'  atc nulo: {invima.atc.isnull().sum()}')
print(f'  estado nulo: {invima.estado.isnull().sum()}')

# Distribución por estado
print(f'\nDistribución por estado:')
print(invima.estado.value_counts().to_string())

In [ ]:
# Cobertura temporal
invima['periodo'] = invima['anio'] * 100 + invima['mes']
cobertura = invima.groupby(['anio', 'mes']).size().reset_index(name='alertas')
cobertura['label'] = cobertura.apply(lambda r: f"{int(r.mes):02d}/{int(r.anio)}", axis=1)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(cobertura)), cobertura['alertas'], color='#f59e0b', edgecolor='white')
ax.set_xticks(range(len(cobertura)))
ax.set_xticklabels(cobertura['label'], rotation=45, fontsize=9)
ax.set_title('Alertas INVIMA por mes (ene 2025 – may 2026)', fontsize=12)
ax.set_ylabel('Número de alertas')
plt.tight_layout()
plt.show()

print(f'Promedio de alertas por mes: {cobertura.alertas.mean():.0f}')
print(f'Mes con más alertas: {cobertura.loc[cobertura.alertas.idxmax(), "label"]} ({cobertura.alertas.max()} alertas)')

## 5. Cruce CUM × INVIMA

La llave de integración es el código ATC. El CUM tiene ATC hasta nivel 7; el INVIMA reporta por principio activo que se mapea a ATC.

In [ ]:
# Productos del CUM que han tenido alerta INVIMA alguna vez
atcs_invima = set(invima['atc'].dropna().str[:5].unique())

cum_con_atc = pd.read_sql("""
    SELECT atc_normalizado, estado_cum
    FROM cum_normalizado
    WHERE atc_normalizado IS NOT NULL AND atc_normalizado != ''
""", conn)

cum_con_atc['atc5'] = cum_con_atc['atc_normalizado'].str[:5]
cum_activos_con_atc = cum_con_atc[cum_con_atc.estado_cum == 'Activo']

en_invima = cum_activos_con_atc['atc5'].isin(atcs_invima)
print(f'Medicamentos CUM activos con ATC en base INVIMA: {en_invima.sum():,} / {len(cum_activos_con_atc):,}')
print(f'Cobertura de cruce: {en_invima.mean()*100:.1f}%')

conn.close()
print('\nLimpieza y transformación documentada. Pipeline completo en backend/etl/')